# Lab | Pandas

In this lab, we will be working with the customer data from an insurance company, which can be found in the CSV file located at the following link: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

The data includes information such as customer ID, state, gender, education, income, and other variables that can be used to perform various analyses.

Throughout the lab, we will be using the pandas library in Python to manipulate and analyze the data. Pandas is a powerful library that provides various data manipulation and analysis tools, including the ability to load and manipulate data from a variety of sources, including CSV files.

### Data Description

- Customer - Customer ID

- ST - State where customers live

- Gender - Gender of the customer

- Education - Background education of customers 

- Customer Lifetime Value - Customer lifetime value(CLV) is the total revenue the client will derive from their entire relationship with a customer. In other words, is the predicted or calculated value of a customer over their entire duration as a policyholder with the insurance company. It is an estimation of the net profit that the insurance company expects to generate from a customer throughout their relationship with the company. Customer Lifetime Value takes into account factors such as the duration of the customer's policy, premium payments, claim history, renewal likelihood, and potential additional services or products the customer may purchase. It helps insurers assess the long-term profitability and value associated with retaining a particular customer.

- Income - Customers income

- Monthly Premium Auto - Amount of money the customer pays on a monthly basis as a premium for their auto insurance coverage. It represents the recurring cost that the insured person must pay to maintain their insurance policy and receive coverage for potential damages, accidents, or other covered events related to their vehicle.

- Number of Open Complaints - Number of complaints the customer opened

- Policy Type - There are three type of policies in car insurance (Corporate Auto, Personal Auto, and Special Auto)

- Vehicle Class - Type of vehicle classes that customers have Two-Door Car, Four-Door Car SUV, Luxury SUV, Sports Car, and Luxury Car

- Total Claim Amount - the sum of all claims made by the customer. It represents the total monetary value of all approved claims for incidents such as accidents, theft, vandalism, or other covered events.


External Resources: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

## Challenge 1: Understanding the data

In this challenge, you will use pandas to explore a given dataset. Your task is to gain a deep understanding of the data by analyzing its characteristics, dimensions, and statistical properties.

- Identify the dimensions of the dataset by determining the number of rows and columns it contains.
- Determine the data types of each column and evaluate whether they are appropriate for the nature of the variable. You should also provide suggestions for fixing any incorrect data types.
- Identify the number of unique values for each column and determine which columns appear to be categorical. You should also describe the unique values of each categorical column and the range of values for numerical columns, and give your insights.
- Compute summary statistics such as mean, median, mode, standard deviation, and quartiles to understand the central tendency and distribution of the data for numerical columns. You should also provide your conclusions based on these summary statistics.
- Compute summary statistics for categorical columns and providing your conclusions based on these statistics.

In [2]:
import pandas as pd
import numpy as np

# The url to dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"

# Import dataset
df = pd.read_csv(url)
df_copy = df.copy()

df.head()
df.info()
df.index

# For data type we can see that "Customer Lifetime value" should be numerical. It's readed as text because of symbol %
# and "Number of Open complaints" it should be numerical/count. It's liely an object bcause of "none" values or formating like 1/0/0
df.dtypes

# -------- Categorical Insights -------
# ST (State): This is a Nominal categorical variable. It helps identify geographic clusters.
# Gender is categorica/nominal, a primary demographic variable often used to analyze risk profiles or claim frequencies.
# Education: This is an Ordinal categorical variable because there is a logical hierarchy (e.g., PhD > Master > Bachelor).
# Number of Open Complaints: While technicaly numerical, it behaves like an Ordinal category in this dataset because the values are discrete and limited (0 to 5). Most customers likely have 0 complaints.
# Policy Type: A Nominal category. Most datasets of this type show "Personal Auto" as the dominant category.
# Vehicle Class is a Nominal Categorical

# ------- Numerical Insights & Ranges ------------
# Customer Lifetime Value (CLV): The wide range suggests a high variance in customer profitability. You should look for "outliers" on the high end (the $80k+ values).
# Income: A significant range ($0 - $99,999). The presence of 0 is a key insight—it likely represents unemployed customers or missing data that needs handling.
# Monthly Premium auto - Since premiums represent currency, they are measurements that require decimal precision.
# Total Claim Amount: The range up to ~$2,893 compared to the Monthly Premium suggests that a single large claim can far exceed years of premium payments.
df.nunique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4008 entries, 0 to 4007
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Customer                   1071 non-null   object 
 1   ST                         1071 non-null   object 
 2   GENDER                     954 non-null    object 
 3   Education                  1071 non-null   object 
 4   Customer Lifetime Value    1068 non-null   object 
 5   Income                     1071 non-null   float64
 6   Monthly Premium Auto       1071 non-null   float64
 7   Number of Open Complaints  1071 non-null   object 
 8   Policy Type                1071 non-null   object 
 9   Vehicle Class              1071 non-null   object 
 10  Total Claim Amount         1071 non-null   float64
dtypes: float64(3), object(8)
memory usage: 344.6+ KB


Customer                     1071
ST                              8
GENDER                          5
Education                       6
Customer Lifetime Value      1027
Income                        774
Monthly Premium Auto          132
Number of Open Complaints       6
Policy Type                     3
Vehicle Class                   6
Total Claim Amount            761
dtype: int64

# Before calculating stat I'll clean all data.

In [3]:
ST_unique = df["ST"].unique()
ST_unique_count = df["ST"].value_counts()

# ------ ST -----
# I'll short all states to two letters. by creating a mapping dictionarry: {"Current Value": "New Value"}
mapping_ST = {
    "Washington": "WA",
    "Arizona": "AZ",
    "Nevada": "NV",
    "California": "CA",
    "Cali": "CA",
    "Oregon": "OR",
}
df["ST"] = df["ST"].replace(mapping_ST)

In [4]:
# ---GENDER ----
# I'll convert Gender to M and F
gender_uniue = df["GENDER"].unique()
gender_unique_count = df["GENDER"].value_counts()

mapping_GENDER = {"Male": "M", "female": "F", "Femal": "F"}
df["GENDER"] = df["GENDER"].replace(mapping_GENDER)

In [5]:
# --- Education ----
# I'll convert from Bachelors to Bachelor
education_unique_count = df["Education"].value_counts()
df["Education"] = df["Education"].replace("Bachelors", "Bachelor")

In [6]:
# ----- Customer Lifetime Value -----
# Remove the '%' character from the strings. Convert the cleaned string to a numeric float
df["Customer Lifetime Value"] = pd.to_numeric(
    df["Customer Lifetime Value"].str.replace("%", ""), errors="coerce"
)
df["Customer Lifetime Value"].dtype

dtype('float64')

In [7]:
# ----------- Number of Open Complaints -------------
# # Extract the second character (the actual number of complaints)
# This assumes the format is always '1/X/00'
df["Number of Open Complaints"] = df["Number of Open Complaints"].apply(
    lambda x: x.split("/")[1] if isinstance(x, str) else x
)

# Convert to numeric, turning any errors or remaining weird text into NaN
df["Number of Open Complaints"] = pd.to_numeric(df["Number of Open Complaints"], errors="coerce")

# Verify the 6 distinct values (0, 1, 2, 3, 4, 5)
print(df["Number of Open Complaints"].unique())

[ 0.  2.  1.  3.  5.  4. nan]


In [8]:
# -------------- Vehicle_Class -------------------
# Not sure how we should change it

Vehicle_Class_unique = df["Vehicle Class"].unique()
Vehicle_Class_unique_count = df["Vehicle Class"].value_counts()

In [9]:
# ------------ Stat for numerical data -------------
# Filter by numerical data
numerical_df = df.select_dtypes(include=["number"])
numerical_stat = round(numerical_df.describe(), 2)

# ------------ Stat for Categorical data -------------
# Filter by categorical data
categorical_df = df.select_dtypes(include=["object"])
categorical_stat = categorical_df.describe()

# ----  Stat interpretation for numerical data -----
## Customer Lifetime Value (CLV)
    This is the most "extreme" column in your dataset.
    The Skew: The Mean ($793,690$) is much higher than the Median ($588,174$). 
    This indicates that a few customers have massive lifetime values that inflate the average.
    The Range: While the minimum is $200,435$, the maximum is a staggering $5,816,655$.
    Insight: Your "typical" customer is worth about $588k, but your marketing team might be focusing on the "average" of $793k, which is unrealistic for 75% of your base.

## Income
    The Spread: Incomes range from $0$ to nearly $100,000$.
    The Middle: The median income is $36,234$, which is quite close to the mean of $39,295$. 
    This suggests that income is more "normally" distributed than CLV, though still slightly tilted toward higher earners.
    The Low End: At least 25% of your customers earn $14,072$ or less.

## Monthly Premium Auto
    The Outlier Problem: Look at the Max ($35,354$) compared to the 75% mark ($109.50$).
    The Reality: 75% of your customers pay less than $110$ a month. 
    However, because of one or more massive premiums (likely commercial or luxury fleets), the Standard Deviation ($1,601$) is huge.
    Interpretation: The "Mean" of $193$ is totally misleading here. The Median ($83$) is the only reliable number to describe what a normal customer pays.

## Total Claim Amount
    Distribution: This follows a similar pattern to CLV. The mean ($404.99$) is higher than the median ($354.73$).
    Concentration: Most claims are relatively small, but the max claim of $2,893$ pushes the average up.

# ---- Stat interpretation for categorical data ----
    ST (State)	    5	    CA is the most frequent state with 331 entries.
    GENDER	        2	    F (Female) is the most common category (502 entries). Note that some entries (117) are missing gender data.
    Education	    5	    Bachelor is the most frequent education level.
    Policy Type	    3	    Personal Auto dominates the dataset (780 out of 1071 rows).
    Vehicle Class	6	    Four-Door Car is the most common vehicle type.

## Challenge 2: analyzing the data

### Exercise 1

The marketing team wants to know the top 5 less common customer locations. Create a pandas Series object that contains the customer locations and their frequencies, and then retrieve the top 5 less common locations in ascending order.

In [10]:
df_customer_location = df["ST"].value_counts(ascending=True)
print(df_customer_location)

ST
NV     98
WA    111
AZ    211
OR    320
CA    331
Name: count, dtype: int64


### Exercise 2

The sales team wants to know the total number of policies sold for each type of policy. Create a pandas Series object that contains the policy types and their total number of policies sold, and then retrieve the policy type with the highest number of policies sold.

*Hint:*
- *Using value_counts() method simplifies this analysis.*
- *Futhermore, there is a method that returns the index of the maximum value in a column or row.*


In [11]:
df_policy_type_counts = df["Policy Type"].value_counts(ascending=False)
top_policy_type = df_policy_type_counts.idxmax()

### Exercise 3

The sales team wants to know if customers with Personal Auto have a lower income than those with Corporate Auto. How does the average income compare between the two policy types?

- Use *loc* to create two dataframes: one containing only Personal Auto policies and one containing only Corporate Auto policies.
- Calculate the average income for each policy.
- Print the results.

In [12]:
# 1. Group by Policy Type and calculate the mean income
income_comparison = round(df.groupby("Policy Type")["Income"].mean(), 2)
print(income_comparison)

# 2. Extract the specific values for comparison
# Create a DataFrame containing only Personal Auto policies
df_personal = df.loc[df["Policy Type"] == "Personal Auto"]
# Create a DataFrame containing only Corporate Auto policies
df_corporate = df.loc[df["Policy Type"] == "Corporate Auto"]

print(f"Personal Auto Median Income: {df_personal['Income'].mean():.2f}")
print(f"Corporate Auto Median Income: {df_corporate['Income'].mean():.2f}")

# 3. Quick check to answer the sales team's question
if df_personal["Income"].mean() < df_corporate["Income"].mean():
    print("\nYes, customers with Personal Auto have a lower average income.")
else:
    print("\nNo, customers with Personal Auto do not have a lower average income.")

Policy Type
Corporate Auto    41390.31
Personal Auto     38180.70
Special Auto      45954.70
Name: Income, dtype: float64
Personal Auto Median Income: 38180.70
Corporate Auto Median Income: 41390.31

Yes, customers with Personal Auto have a lower average income.


### Bonus: Exercise 4


Your goal is to identify customers with a high policy claim amount.

Instructions:

- Review again the statistics for total claim amount to gain an understanding of the data.
- To identify potential areas for improving customer retention and profitability, we want to focus on customers with a high policy claim amount. Consider customers with a high policy claim amount to be those in the top 25% of the total claim amount. Create a pandas DataFrame object that contains information about customers with a policy claim amount greater than the 75th percentile.
- Use DataFrame methods to calculate summary statistics about the high policy claim amount data. 

*Note: When analyzing data, we often want to focus on certain groups of values to gain insights. Percentiles are a useful tool to help us define these groups. A percentile is a measure that tells us what percentage of values in a dataset are below a certain value. For example, the 75th percentile represents the value below which 75% of the data falls. Similarly, the 25th percentile represents the value below which 25% of the data falls. When we talk about the top 25%, we are referring to the values that fall above the 75th percentile, which represent the top quarter of the data. On the other hand, when we talk about the bottom 25%, we are referring to the values that fall below the 25th percentile, which represent the bottom quarter of the data. By focusing on these groups, we can identify patterns and trends that may be useful for making decisions and taking action.*

*Hint: look for a method that gives you the percentile or quantile 0.75 and 0.25 for a Pandas Series.*

*Hint 2: check `Boolean selection according to the values of a single column` in https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9*

In [ ]:
df_claim = df["Total Claim Amount"].describe().round(2)
threshold = df["Total Claim Amount"].quantile(0.75)
print(f"\nThe 75th percentile threshold is: {threshold:.2f}")

# Create the filtered DataFrame
df_high_calims = df[df["Total Claim Amount"] > threshold].copy()

# Calculate the stat for the high-claim subset
high_claim_stat = round(df_high_calims.describe(), 2)


The 75th percentile threshold is: 532.80
